# Complex cashflows (foundation for bonds and loans)

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`. We map **cashflow engineering** ideas to `bond` JSON (`cashflow_spec`) and price a **linearly amortizing** note.


## Concept

- **Amortization:** bullet, linear paydown, custom schedules.
- **PIK vs cash:** changes outstanding principal paths.
- **Floaters:** caps/floors; **calls/puts** reshape horizon risk.

Deposits remain the simplest pipeline check; bonds carry richer schedules.


In [1]:
import json
from datetime import date

from finstack_quant.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import DiscountCurve, MarketContext

print("Imports OK (complex cashflows).")


Imports OK (complex cashflows).


## Instrument JSON


In [2]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/complex_cashflows.json").read_text())

AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

bond_amort = _NOTEBOOK_DATA['bond_amort']
bond_json = json.dumps(bond_amort)
try:
    print("validate prefix:", validate_instrument_json(bond_json)[:180], "…")
except Exception as exc:
    print("validate failed:", type(exc).__name__, exc)


validate prefix: {
  "spec": {
    "attributes": {},
    "call_put": null,
    "cashflow_spec": {
      "Amortizing": {
        "base": {
          "Fixed": {
            "bdc": "following",
       …


## MarketContext


In [3]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
mc = MarketContext().insert(ois)
market_json = mc.to_json()
print("market_json chars:", len(market_json))


market_json chars: 977


## Pricing


In [4]:
try:
    out = price_instrument_with_metrics(bond_json, market_json, AS_OF_STR, model="discounting")
    vr = ValuationResult.from_json(out)
    print(vr)
except Exception as exc:
    print("price failed:", type(exc).__name__, exc)


ValuationResult(id="AMORT-FIXED", price=851271.6212, currency=USD, metrics=0)


## Metrics


In [5]:
try:
    out = price_instrument_with_metrics(
        bond_json, market_json, AS_OF_STR, model="discounting",
        metrics=["dv01", "duration_mod", "convexity"],
    )
    vr = ValuationResult.from_json(out)
    print("dv01:", vr.get_metric("dv01"))
    print("duration_mod:", vr.get_metric("duration_mod"))
    print("convexity:", vr.get_metric("convexity"))
except Exception as exc:
    print("metrics failed:", type(exc).__name__, exc)


dv01: -215.54724364948925
duration_mod: 2.488654250705142
convexity: 0.0895282539964533


## Cashflows

After pricing, extract the cashflow schedule that produced the PV using `instrument_cashflows` (returns `(envelope, DataFrame)`) or the lower-level `instrument_cashflows_json`.


In [6]:
from finstack_quant.valuations import instrument_cashflows, instrument_cashflows_json

env, df = instrument_cashflows(bond_json, market_json, AS_OF_STR, model="discounting")
print("total_pv from envelope:", env.get("total_pv"))
print("flows (head):")
print(df.head().to_string())

# Also the raw JSON form (for round-tripping or other consumers)
cf_json = instrument_cashflows_json(bond_json, market_json, AS_OF_STR, model="discounting")
print("\ninstrument_cashflows_json length:", len(cf_json), "chars")

total_pv from envelope: 851271.621158788
flows (head):
        date   amount currency          kind  accrual_factor  year_fraction  rate  discount_factor            pv
0 2025-01-15  18400.0      USD         Fixed             0.5        0.00000  0.04         1.000000      0.000000
1 2025-01-15  80000.0      USD  Amortization             0.0        0.00000   NaN         1.000000      0.000000
2 2025-07-15  16800.0      USD         Fixed             0.5        0.49589  0.04         0.985123  16550.071393
3 2025-07-15  80000.0      USD  Amortization             0.0        0.49589   NaN         0.985123  78809.863776
4 2026-01-15  15200.0      USD         Fixed             0.5        1.00000  0.04         0.970000  14744.000000

instrument_cashflows_json length: 3716 chars


## Direct schedule construction (`finstack_quant.cashflows`)

The `cashflows` module exposes the JSON-first schedule engine independent of any instrument: build a canonical schedule from a `CashflowScheduleBuildSpec` (`build_cashflow_schedule_json`), wrap it with build metadata (`build_cashflow_schedule_envelope_json`), validate either form, extract `dated_flows_json`, compute `accrued_interest_json`, and wrap a schedule as a priceable bond via `bond_from_cashflows_json`.

In [ ]:
from finstack_quant.cashflows import (
    accrued_interest_json,
    bond_from_cashflows_json,
    build_cashflow_schedule_envelope_json,
    build_cashflow_schedule_json,
    dated_flows_json,
    validate_cashflow_schedule_envelope_json,
    validate_cashflow_schedule_json,
)

cf_spec = json.dumps({
    "notional": {"initial": {"amount": "1000000", "currency": "USD"}, "amort": "None"},
    "issue": "2024-08-31",
    "maturity": "2025-08-31",
    "fixed_coupons": [{
        "coupon_type": "Cash", "rate": "0.06",
        "freq": {"count": 12, "unit": "months"},
        "dc": "Thirty360", "bdc": "following", "calendar_id": "weekends_only",
        "stub": "None", "end_of_month": False, "payment_lag_days": 0,
    }],
})

schedule_json = build_cashflow_schedule_json(cf_spec)
validate_cashflow_schedule_json(schedule_json)
schedule = json.loads(schedule_json)
print("flows:", len(schedule["flows"]), "| issue:", schedule["meta"]["issue_date"])

envelope_json = build_cashflow_schedule_envelope_json(cf_spec)
validate_cashflow_schedule_envelope_json(envelope_json)
print("envelope schema:", json.loads(envelope_json)["schema_version"])

flows = json.loads(dated_flows_json(schedule_json))
print("dated flows:", len(flows))
print("accrued @ 2025-02-28:", round(accrued_interest_json(schedule_json, "2025-02-28"), 2))

inst_json = bond_from_cashflows_json("CUSTOM-CF", schedule_json, "USD-OIS", 99.0)
print("bond_from_cashflows type:", json.loads(inst_json)["type"])

## Takeaways

- **Bond `cashflow_spec`** encodes the same ideas as loan schedules in many systems.
- **Validate first**, then price; schedule errors are easier to read from validation.
- **Deposits** are the minimal alternative when you only need PV off one curve.
